In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [3]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [5]:
df.shape

(898, 35)

In [6]:
X = df.drop(["Class"], axis=1)
y = df["Class"]

In [7]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### ANN

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [11]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=float)
y_train_tensor = torch.tensor(y_train, dtype=float)

X_test_tensor = torch.tensor(X_test_scaled, dtype=float)
y_test_tensor = torch.tensor(y_test, dtype=float)

In [12]:
X_train_tensor = X_train_tensor.float()
y_train_tensor = y_train_tensor.long()
X_test_tensor = X_test_tensor.float()
y_test_tensor = y_test_tensor.long()

In [13]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [14]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [15]:
# Building the model
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64), 
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 7)
        )

    def forward(self, x):
        return self.model(x)

In [16]:
model = ANN()

# loss and optim
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [17]:
# Training the NN

epochs = 100

for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    print(f"Epoch : {epoch + 1}/{epochs}, loss = {train_loss}")

Epoch : 1/100, loss = 1.7207375557526299
Epoch : 2/100, loss = 1.0974414322687231
Epoch : 3/100, loss = 0.6999795242496158
Epoch : 4/100, loss = 0.5446838244147922
Epoch : 5/100, loss = 0.4628270903359289
Epoch : 6/100, loss = 0.4026333158430846
Epoch : 7/100, loss = 0.3714409269716429
Epoch : 8/100, loss = 0.3384970996690833
Epoch : 9/100, loss = 0.3048335760831833
Epoch : 10/100, loss = 0.2797203200018924
Epoch : 11/100, loss = 0.2619041146143623
Epoch : 12/100, loss = 0.25299043111179187
Epoch : 13/100, loss = 0.2294528706242209
Epoch : 14/100, loss = 0.22389131719651428
Epoch : 15/100, loss = 0.21088564136753912
Epoch : 16/100, loss = 0.20534972360600595
Epoch : 17/100, loss = 0.1941901382220828
Epoch : 18/100, loss = 0.18725051123486913
Epoch : 19/100, loss = 0.17650870541515556
Epoch : 20/100, loss = 0.18213090300559998
Epoch : 21/100, loss = 0.1756435124122578
Epoch : 22/100, loss = 0.15782993659377098
Epoch : 23/100, loss = 0.15428165605534677
Epoch : 24/100, loss = 0.147669842

In [20]:
# Evaluation
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0)

print("Accuracy =", correct / total *100)

Accuracy = 91.66666666666666
